# Week 4, The Data Cookbook: Load a File, an API, a Polite Scrape

**What you'll do today.** Three ways to get a corpus: the **API** (the polite front
door) and a small, careful **scrape** (the back door, used with care).

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q requests beautifulsoup4 pandas datasets gdown

In [ ]:
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
print("imports ok")

In [ ]:
# Are we running tiny + offline (the compatibility harness sets this), or in a real
# Colab class session? Everything below adapts so the notebook runs either way.
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Route 1, load a prepared file (the most common case)

Most corpora already exist as a file somewhere: a CSV on GitHub, a dataset on Hugging Face, a
zip in a Drive folder. This is the route you'll use most, and it's one line:

- a direct link: `pd.read_csv(url)`
- a Hugging Face dataset: `datasets.load_dataset(...)`, then `.to_pandas()`
- a file in your Drive: point pandas at the Drive path (you mounted it in the first cell)
- a zip or archive: download, `unzip`, then read the file inside

**Music note:** Spotify's audio-features API was shut off in Nov 2024. For music, load a
Billboard Hot 100 + audio-features CSV instead of calling Spotify.

In [ ]:
import io

# A tiny CSV stands in for a real file so this always runs. In class, set `url` to a real
# direct link (a GitHub "raw" CSV, a dataset export, your own file) and re-run.
SAMPLE_CSV = "title,year,medium\nBlue Vase,1901,ceramic\nRed Bowl,1923,ceramic\nGreen Jug,1947,glass\n"
url = None  # e.g. "https://raw.githubusercontent.com/<user>/<repo>/main/data.csv"

if url and not SMOKE:
    prepared = pd.read_csv(url)
else:
    prepared = pd.read_csv(io.StringIO(SAMPLE_CSV))
print(prepared.head())

# Other one-liners the AI can write for you (uncomment the one you need):
# from datasets import load_dataset                 # Hugging Face
# prepared = load_dataset("some/dataset")["train"].to_pandas()
# import gdown; gdown.download(url, f"{PROJECT_DIR}/data.csv", quiet=True)
# !wget -q <url> -O "$PROJECT_DIR/data.csv"          # any direct link into Drive
# !unzip -o archive.zip -d "$PROJECT_DIR"            # then pd.read_csv the file inside

## Route 2, the API (the polite front door)

An **endpoint** is a URL that returns *data* instead of a web page.

In [ ]:
AIC = "https://api.artic.edu/api/v1/artworks?limit=5&fields=id,title,date_display,artist_title"

# A tiny built-in sample so the notebook runs even with no network (the harness uses this).
SAMPLE = {"data": [
    {"id": 1, "title": "The Bedroom", "date_display": "1889", "artist_title": "Vincent van Gogh"},
    {"id": 2, "title": "A Sunday on La Grande Jatte", "date_display": "1884-86", "artist_title": "Georges Seurat"},
    {"id": 3, "title": "Nighthawks", "date_display": "1942", "artist_title": "Edward Hopper"},
]}

def get_json(url):
    if SMOKE:
        return SAMPLE
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "culture-as-data-course"})
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print("network call failed, using sample:", e)
        return SAMPLE

payload = get_json(AIC)
df = pd.json_normalize(payload["data"])
print(df.to_string(index=False))

A documented URL became a table.

## Route 3, a polite scrape (the back door)

When there's no API, the AI writes a small `BeautifulSoup` scrape of a *simple* page.

In [ ]:
# We parse a built-in HTML string so the lesson never depends on a live site.
HTML = """
<html><body>
  <ul class="catalog">
    <li class="item"><span class="name">Blue Vase</span><span class="year">1901</span></li>
    <li class="item"><span class="name">Red Bowl</span><span class="year">1923</span></li>
    <li class="item"><span class="name">Green Jug</span><span class="year">1947</span></li>
  </ul>
</body></html>
"""
soup = BeautifulSoup(HTML, "html.parser")
rows = []
for li in soup.select("li.item"):
    rows.append({"name": li.select_one(".name").text, "year": li.select_one(".year").text})
scraped = pd.DataFrame(rows)
print(scraped.to_string(index=False))

## Reshape with the AI; you judge what came back

The AI wrangles columns, types, missing values.

In [ ]:
scraped["year"] = pd.to_numeric(scraped["year"], errors="coerce")
scraped["century"] = (scraped["year"] // 100 + 1).astype("Int64")
print(scraped.to_string(index=False))
print("\nIs this the corpus you wanted? Right rows, right columns, nothing silently dropped?")

In [ ]:
# Save your collected corpus to the Drive project folder so it is here next week.
import os
_out = os.path.join(PROJECT_DIR, "week04_corpus.csv")
scraped.to_csv(_out, index=False)
print("saved corpus ->", _out)

## Ready-made loaders for the common cases

Before writing a collector from scratch, check `notebooks/social-media-starters/`: Reddit
(historical archives, the official API via PRAW, and a fiction loader for r/nosleep and
r/HFY), the Bluesky firehose, Mastodon public timelines, Project Gutenberg, and a polite
scraper template with the delay and checklist built in. Each runs offline on a sample, so
you can read the pipeline before you commit to it.

## You collected a corpus

You pulled data three ways and reshaped it.

**Sketch:** write your **Data Biography** (~400 words) and *actually collect your corpus*.